# 14. Scaled GraphRAG-Bench, Both Domains: Vector RAG vs Graph RAG on All 4,072 Questions

**Presentation map:** Slide 33 (graph construction: chunk → LLM extracts entities and relations), Slide 34 (the
indexing price), Slide 36 (graphs help on reasoning and summarisation, not on fact lookup — source: *When to use
Graphs in RAG*, arXiv 2506.05690), Slide 38 (decision rule by question type).

Notebook 9 tested slide 36 on 180 medical questions and found **no resolvable difference** between vector and graph
RAG. Two doubts remained: the sample was small, and one domain (patient guidelines) may simply suit vector search.
This notebook removes both doubts: it runs **every question of GraphRAG-Bench** (`GraphRAG-Bench/GraphRAG-Bench`, MIT)
in **both domains**:

| Domain | Corpus | Questions (Fact / Reasoning / Summarize / Creative) |
|---|---|---|
| **Medical** | NCCN-style patient guidelines, ≈1.05 M characters | 2,062 |
| **Novel** | 20 public-domain novels, ≈4.8 M characters | 2,010 (971 / 610 / 362 / 67) |

**The pipeline is notebook 9's, unchanged in design:** ~600-token chunks; (a) vector RAG = top-8 chunks by
`text-embedding-3-small`; (b) graph RAG (HippoRAG-2-style) = LLM entity/relation extraction per chunk → entity–chunk
graph with provenance → question entities linked by embedding → personalised PageRank → top-5 chunks + 25 relation
statements. **Scoring** is the benchmark's own *answer correctness* (0.75 × statement-level F1 judged by the LLM +
0.25 × embedding similarity) for every question, plus *coverage* for Summarize/Creative.

**Model:** `gpt-6-luna` (no reasoning) for extraction, answers and the judge, via the OpenAI Batch API
(`smash_batch.py`). *Note:* notebook 9 used `gpt-4o-mini`; compare the two notebooks' **differences** (graph − vector),
not their absolute scores. Cached in `data/benchmarks/graphrag_bench/cache_luna/`.

In [1]:
import json, os, re, time, random
from collections import Counter, defaultdict
import numpy as np
import scipy.sparse as sp
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, paired_boot

MODE = mode_from_env()
TEST = MODE == "test"
BASE = "data/benchmarks/graphrag_bench"
llm = LLM(f"{BASE}/cache_luna" + ("_test" if TEST else ""))
TYPES = ["Fact Retrieval", "Complex Reasoning", "Contextual Summarize", "Creative Generation"]

D = {}
for dom in ("medical", "novel"):
    docs = json.load(open(f"{BASE}/Datasets/Corpus/{dom}.json"))
    qs = sorted(json.load(open(f"{BASE}/Datasets/Questions/{dom}_questions.json")), key=lambda q: q["id"])
    if TEST:                                         # 3 questions per type from the first two sources
        srcs = sorted({q.get("source", "") for q in qs})[:2] if dom == "novel" else None
        docs = [d for d in docs if srcs is None or d.get("corpus_name") in srcs]
        qs = [q for t in TYPES for q in [x for x in qs if x["question_type"] == t and (srcs is None or x.get("source") in srcs)][:3]]
    D[dom] = {"docs": docs, "questions": qs}
    print(f"{dom}: {len(docs)} document(s), {sum(len(d['context']) for d in docs):,} characters, {len(qs):,} questions",
          dict(Counter(q["question_type"] for q in qs)))

medical: 1 document(s), 1,052,159 characters, 2,062 questions {'Contextual Summarize': 289, 'Complex Reasoning': 509, 'Fact Retrieval': 1098, 'Creative Generation': 166}
novel: 20 document(s), 4,819,610 characters, 2,010 questions {'Contextual Summarize': 362, 'Complex Reasoning': 610, 'Fact Retrieval': 971, 'Creative Generation': 67}


## Chunks and the vector index

In [2]:
CHUNK_CHARS, OVERLAP = 2400, 300

def make_chunks(text):
    out, i = [], 0
    while i < len(text):
        j = min(len(text), i + CHUNK_CHARS)
        if j < len(text):
            k = text.rfind(". ", i + CHUNK_CHARS // 2, j)
            j = k + 1 if k > 0 else j
        out.append(text[i:j].strip())
        if j >= len(text):
            break
        i = j - OVERLAP
    return out

for dom, d in D.items():
    d["chunks"] = [c for doc in d["docs"] for c in make_chunks(doc["context"])]
    if TEST and dom == "medical":
        d["chunks"] = d["chunks"][:60]
    d["C"] = llm.embed(f"{dom}_chunks", d["chunks"])
    d["Q"] = llm.embed(f"{dom}_questions", [q["question"] for q in d["questions"]])
    print(f"{dom}: {len(d['chunks']):,} chunks")

medical: 519 chunks
novel: 2,434 chunks


## Graph index: one extraction request per chunk (one Batch job per domain)

In [3]:
class Entity(BaseModel):
    name: str
    type: str
    description: str = Field(description="one sentence, grounded in the text")

class Relation(BaseModel):
    source: str
    target: str
    description: str = Field(description="short phrase stating how source relates to target, grounded in the text")

class Extraction(BaseModel):
    entities: list[Entity]
    relations: list[Relation]

EXTRACT_SYS = {
    "medical": ("Extract a knowledge graph from a medical patient-guideline passage. List the important entities "
                "(diseases, treatments, drugs, tests, symptoms, risk factors, procedures, organisations, statistics) and "
                "the factual relations between them. Use the most specific canonical name; only state what the text "
                "says. At most 25 entities and 25 relations."),
    "novel": ("Extract a knowledge graph from a passage of a novel. List the important entities (characters, places, "
              "objects, organisations, events, dates) and the relations between them stated in the passage (family, "
              "actions, locations, possessions, feelings, causes). Use each character's fullest name; only state what "
              "the text says. At most 25 entities and 25 relations.")}

norm = lambda s: re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

for dom, d in D.items():
    res = llm.run(f"{dom}_extract", {f"c{i}": LLM.body([{"role": "system", "content": EXTRACT_SYS[dom]},
                                                         {"role": "user", "content": c}], max_tokens=4000, schema=Extraction)
                                      for i, c in enumerate(d["chunks"])}, parse=Extraction)
    exs = [res[f"c{i}"]["parsed"] or {"entities": [], "relations": []} for i in range(len(d["chunks"]))]
    n_c = len(d["chunks"])
    ent, ent_name, rels = {}, [], defaultdict(list)
    rows, cols = [], []
    def eid(name):
        k = norm(name)
        if not k:
            return None
        if k not in ent:
            ent[k] = n_c + len(ent_name); ent_name.append(name)
        return ent[k]
    for ci, ex in enumerate(exs):
        for e in ex["entities"]:
            j = eid(e["name"])
            if j is not None:
                rows += [ci, j]; cols += [j, ci]
        for r in ex["relations"]:
            a, b = eid(r["source"]), eid(r["target"])
            if a is None or b is None or a == b:
                continue
            rows += [a, b, ci, a, ci, b]; cols += [b, a, a, ci, b, ci]
            rels[(min(a, b), max(a, b))].append((r["description"], r["source"], r["target"], ci))
    n = n_c + len(ent_name)
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n, n)); A.data[:] = 1
    deg = np.asarray(A.sum(1)).ravel(); deg[deg == 0] = 1
    rel_by_ent = defaultdict(list)
    for a, b in rels:
        rel_by_ent[a].append((a, b)); rel_by_ent[b].append((a, b))
    d.update(T=(sp.diags(1 / deg) @ A).T.tocsr(), n_c=n_c, ent_name=ent_name, rels=rels, rel_by_ent=rel_by_ent,
             E=llm.embed(f"{dom}_entities", ent_name, cap=200),
             unparsed=sum(res[f"c{i}"]["parsed"] is None for i in range(n_c)))
    print(f"{dom}: {n_c:,} extraction requests ({d['unparsed']} unparsable) -> {len(ent_name):,} entities, "
          f"{len(rels):,} related entity pairs")

medical: 519 extraction requests (0 unparsable) -> 4,835 entities, 8,264 related entity pairs


novel: 2,434 extraction requests (0 unparsable) -> 28,266 entities, 38,141 related entity pairs


## Retrieval: vector top-8 vs graph (personalised PageRank → top-5 chunks + 25 relations)

The question's entities are extracted by the LLM (one Batch job), each is linked to its 3 nearest graph entities
(cosine > 0.5), and a small weight goes to the 20 best dense chunks (HippoRAG 2).

In [4]:
class QueryEntities(BaseModel):
    entities: list[str]

QENT_SYS = {"medical": "List the medical entities (diseases, treatments, tests, symptoms, risk factors, ...) mentioned or implied in the question.",
            "novel": "List the entities (characters, places, objects, events, works) mentioned or implied in the question."}

def ppr(T, pers, alpha=0.5, iters=40):
    p = pers / pers.sum(); x = p.copy()
    for _ in range(iters):
        x = (1 - alpha) * p + alpha * (T @ x)
    return x

for dom, d in D.items():
    qe = llm.run(f"{dom}_query_entities", {q["id"]: LLM.body([{"role": "system", "content": QENT_SYS[dom]},
                                                               {"role": "user", "content": q["question"]}],
                                                              max_tokens=200, schema=QueryEntities)
                                            for q in d["questions"]}, parse=QueryEntities)
    q_ents = [(qe[q["id"]]["parsed"] or {"entities": []})["entities"] or [q["question"]] for q in d["questions"]]
    flat = sorted({e for es in q_ents for e in es})
    QE = dict(zip(flat, llm.embed(f"{dom}_query_entity_vecs", flat, cap=200)))
    d["vec"], d["graph"] = [], []
    for qi, q in enumerate(d["questions"]):
        dense = d["C"] @ d["Q"][qi]
        top8 = list(np.argsort(-dense)[:8])
        d["vec"].append(("\n\n".join(f"[chunk {i}] {d['chunks'][i]}" for i in top8), top8))
        pers = np.zeros(d["T"].shape[0])
        for e in q_ents[qi]:
            sims = d["E"] @ QE[e]
            for j in np.argsort(-sims)[:3]:
                if sims[j] > 0.5:
                    pers[d["n_c"] + j] += sims[j]
        for j in np.argsort(-dense)[:20]:
            pers[j] += 0.05 * max(dense[j], 0)
        x = ppr(d["T"], pers)
        top_c = list(np.argsort(-x[:d["n_c"]])[:5])
        top_e = set(d["n_c"] + np.argsort(-x[d["n_c"]:])[:40])
        pairs = {pr for e in top_e for pr in d["rel_by_ent"].get(e, ()) if pr[0] in top_e and pr[1] in top_e}
        cand = [(x[a] + x[b], f"{s} — {desc} — {t}") for (a, b) in pairs for desc, s, t, _ in d["rels"][(a, b)][:2]]
        rel_lines = [r for _, r in sorted(cand, reverse=True)[:25]]
        ctx = ("Relations from the knowledge graph:\n" + "\n".join(f"- {r}" for r in rel_lines) + "\n\n" +
               "\n\n".join(f"[chunk {i}] {d['chunks'][i]}" for i in top_c))
        d["graph"].append((ctx, top_c))
    print(f"{dom}: retrieval done for {len(d['questions']):,} questions")

medical: retrieval done for 2,062 questions


novel: retrieval done for 2,010 questions


## Answers (one Batch job per domain)

In [5]:
ANSWER_SYS = {"medical": "You answer questions about medical patient guidelines using only the provided context. Answer directly and completely; do not mention the context.",
              "novel": "You answer questions about novels using only the provided context. Answer directly and completely; do not mention the context."}
for dom, d in D.items():
    reqs = {f"{m}|{q['id']}": LLM.body([{"role": "system", "content": ANSWER_SYS[dom]},
                                        {"role": "user", "content": f"Context:\n{d[m][qi][0]}\n\nQuestion: {q['question']}"}],
                                       max_tokens=1000)
            for qi, q in enumerate(d["questions"]) for m in ("vec", "graph")}
    d["ans"] = {k: v["text"] for k, v in llm.run(f"{dom}_answers", reqs).items()}
    print(f"{dom}: {len(reqs):,} answers")

medical: 4,124 answers


novel: 4,018 answers


## Scoring with the benchmark's metrics (two Batch rounds: statements, then classification and coverage)

In [6]:
class Statements(BaseModel):
    statements: list[str]

class Classified(BaseModel):
    TP: list[str]
    FP: list[str]
    FN: list[str]

class Coverage(BaseModel):
    covered: list[int] = Field(description="1 if the fact is covered by the response, else 0, one per reference fact, in order")

STMT = "Break the text into concise, independent factual statements.\n\n"
for dom, d in D.items():
    texts = {f"gold|{q['id']}": q["answer"] for q in d["questions"]}
    texts.update({k: v for k, v in d["ans"].items()})
    st = llm.run(f"{dom}_statements", {k: LLM.body([{"role": "user", "content": STMT + t}], max_tokens=2000, schema=Statements)
                                       for k, t in texts.items()}, parse=Statements)
    d["st"] = {k: (v["parsed"] or {"statements": []})["statements"] for k, v in st.items()}
    reqs = {}
    for q in d["questions"]:
        gold = d["st"][f"gold|{q['id']}"]
        for m in ("vec", "graph"):
            a = d["st"][f"{m}|{q['id']}"]
            reqs[f"cls|{m}|{q['id']}"] = LLM.body([{"role": "user", "content":
                "Compare answer statements with ground-truth statements. TP: answer statement supported by the "
                "ground truth. FP: answer statement not supported by (or contradicting) the ground truth. FN: "
                "ground-truth statement missing from the answer. Return the statements in each list.\n\n"
                f"Question: {q['question']}\nAnswer statements: {json.dumps(a)}\nGround-truth statements: {json.dumps(gold)}"}],
                max_tokens=3000, schema=Classified)
            if q["question_type"] in ("Contextual Summarize", "Creative Generation") and gold:
                reqs[f"cov|{m}|{q['id']}"] = LLM.body([{"role": "user", "content":
                    "For each reference fact, is it covered by the response? Answer 1 or 0 per fact, in order.\n\n"
                    f"Question: {q['question']}\nResponse: {d['ans'][f'{m}|{q['id']}'][:3000]}\nReference facts: {json.dumps(gold)}"}],
                    max_tokens=500, schema=Coverage)
    jr = llm.run(f"{dom}_judge", reqs)
    d["judge"] = {}
    for k, v in jr.items():
        try:
            d["judge"][k] = json.loads(v["text"])
        except Exception:
            d["judge"][k] = None
    # semantic similarity part of answer correctness
    keys = [f"{m}|{q['id']}" for q in d["questions"] for m in ("vec", "graph")]
    A = llm.embed(f"{dom}_answer_vecs", [d["ans"][k] for k in keys])
    G = llm.embed(f"{dom}_gold_vecs", [q["answer"] for q in d["questions"]])
    d["sim"] = {k: float(A[i] @ G[i // 2]) for i, k in enumerate(keys)}
    print(f"{dom}: {len(st):,} statement requests, {len(reqs):,} judge requests")

def correctness(d, m, q):
    gold = d["st"][f"gold|{q['id']}"]; a = d["st"][f"{m}|{q['id']}"]
    c = d["judge"].get(f"cls|{m}|{q['id']}")
    if not a and not gold:
        f1 = 1.0
    elif not c:
        f1 = 0.0
    else:
        tp, fp, fn = len(c["TP"]), len(c["FP"]), len(c["FN"])
        p, r = tp / (tp + fp + 1e-10), tp / (tp + fn + 1e-10)
        f1 = 2 * p * r / (p + r + 1e-10)
    return 0.75 * f1 + 0.25 * (d["sim"][f"{m}|{q['id']}"] + 1) / 2

def coverage(d, m, q):
    gold = d["st"][f"gold|{q['id']}"]; c = d["judge"].get(f"cov|{m}|{q['id']}")
    if not gold or not c:
        return float("nan")
    cv = (c["covered"] + [0] * len(gold))[:len(gold)]
    return sum(1 for x in cv if x) / len(gold)

medical: 6,186 statement requests, 5,034 judge requests


novel: 6,027 statement requests, 4,876 judge requests


## Results by domain and question type, with 95% paired-bootstrap intervals

In [7]:
SUMMARY = {}
print(f"{'domain':<9}{'question type':<23}{'n':>6}{'vector':>9}{'graph':>8}{'graph - vector [95% CI]':>27}{'coverage v / g':>17}")
for dom, d in D.items():
    for t in TYPES + ["ALL"]:
        qs = [q for q in d["questions"] if t == "ALL" or q["question_type"] == t]
        if not qs:
            continue
        v = [correctness(d, "vec", q) for q in qs]; g = [correctness(d, "graph", q) for q in qs]
        dm, lo, hi = paired_boot(v, g)
        cov = ""
        if t in ("Contextual Summarize", "Creative Generation"):
            cv = np.nanmean([coverage(d, "vec", q) for q in qs]); cg = np.nanmean([coverage(d, "graph", q) for q in qs])
            cov = f"{cv:.0%} / {cg:.0%}"
        SUMMARY[(dom, t)] = (len(qs), np.mean(v), np.mean(g), dm, lo, hi)
        print(f"{dom:<9}{t:<23}{len(qs):>6}{100 * np.mean(v):>9.1f}{100 * np.mean(g):>8.1f}"
              f"{f'{100 * dm:+.1f} [{100 * lo:+.1f}, {100 * hi:+.1f}]':>27}{cov:>17}")

# Did retrieval reach the gold evidence? (fuzzy: >= 60% of an evidence sentence's words in a retrieved chunk)
def ev_hit(d, ev, ids):
    toks = set(re.findall(r"\w+", ev.lower()))
    return any(len(toks & set(re.findall(r"\w+", d["chunks"][i].lower()))) >= 0.6 * len(toks) for i in ids)
print("\nGold-evidence recall of the retrieved chunks (vector top-8 / graph top-5):")
for dom, d in D.items():
    for t in TYPES:
        vr, gr = [], []
        for qi, q in enumerate(d["questions"]):
            evs = q.get("evidence") or []
            evs = evs if isinstance(evs, list) else [evs]
            if q["question_type"] != t or not evs:
                continue
            vr.append(np.mean([ev_hit(d, e, d["vec"][qi][1]) for e in evs]))
            gr.append(np.mean([ev_hit(d, e, d["graph"][qi][1]) for e in evs]))
        if vr:
            print(f"  {dom:<8} {t:<22} vector {np.mean(vr):5.1%}   graph {np.mean(gr):5.1%}")

domain   question type               n   vector   graph    graph - vector [95% CI]   coverage v / g
medical  Fact Retrieval           1098     64.9    63.3          -1.6 [-3.2, -0.0]                 
medical  Complex Reasoning         509     55.0    53.8          -1.1 [-3.2, +0.9]                 
medical  Contextual Summarize      289     57.6    56.0          -1.6 [-4.0, +0.8]        81% / 69%
medical  Creative Generation       166     45.5    43.6          -1.9 [-4.1, +0.3]        47% / 38%
medical  ALL                      2062     59.9    58.4          -1.5 [-2.5, -0.4]                 
novel    Fact Retrieval            971     58.6    54.7          -3.9 [-6.0, -1.9]                 
novel    Complex Reasoning         610     45.8    48.7          +2.8 [+1.3, +4.4]                 
novel    Contextual Summarize      362     51.4    52.7          +1.3 [-0.9, +3.4]        89% / 88%
novel    Creative Generation        67     30.3    32.5          +2.1 [-1.4, +5.5]        32% / 29%


  medical  Fact Retrieval         vector 97.9%   graph 92.0%


  medical  Complex Reasoning      vector 91.2%   graph 84.2%


  medical  Contextual Summarize   vector 92.2%   graph 82.8%


  medical  Creative Generation    vector 60.5%   graph 46.1%


  novel    Fact Retrieval         vector 91.6%   graph 79.8%
  novel    Complex Reasoning      vector 93.1%   graph 85.8%


  novel    Contextual Summarize   vector 84.6%   graph 75.6%
  novel    Creative Generation    vector 66.6%   graph 52.9%


## The indexing price

In [8]:
LEDGER = llm.print_ledger()
for dom in D:
    ex, an = LEDGER.get(f"{dom}_extract", {}), LEDGER.get(f"{dom}_answers", {})
    print(f"{dom}: graph index {ex.get('calls', 0):,} LLM calls, {(ex.get('in', 0) + ex.get('out', 0)) / 1e6:.1f}M tokens "
          f"(${ex.get('usd', 0):.2f}); the vector index needs only embeddings")

stage                           calls     input tok   output tok      USD  via Batch
medical_answers                 4,124    14,579,391      391,458     0.83       100%
medical_answers_hippo           2,062     8,245,830      200,303     0.46       100%
medical_extract                   519       389,686      630,516     0.18       100%
medical_judge                   5,034     1,667,428      564,515     0.22       100%
medical_query_entities          2,062       195,112      107,031     0.04       100%
medical_statements              6,186       803,607      538,898     0.17       100%
medical_statements_hippo        2,062       311,651      219,254     0.07       100%
novel_answers                   4,018    15,110,052      268,870     0.82       100%
novel_extract                   2,434     1,948,272    2,412,639     0.70       100%
novel_judge                     4,876     1,395,012      396,954     0.17       100%
novel_query_entities            2,009       206,402       70,390 

In [9]:
PAPER = {"Fact Retrieval": (60.92, 49.29), "Complex Reasoning": (42.93, 53.38), "Contextual Summarize": (51.30, 64.40)}
print("=" * 96)
print(f"RESULTS SUMMARY — Notebook 14: GraphRAG-Bench, all questions, both domains, gpt-6-luna, answer correctness x100")
print("=" * 96)
print(f"{'domain':<9}{'question type':<23}{'n':>6}{'vector':>9}{'graph':>8}{'graph - vector [95% CI]':>27}   verdict")
for (dom, t), (n, v, g, dm, lo, hi) in SUMMARY.items():
    verdict = "graph better" if lo > 0 else "vector better" if hi < 0 else "no resolvable difference"
    print(f"{dom:<9}{t:<23}{n:>6}{100 * v:>9.1f}{100 * g:>8.1f}{f'{100 * dm:+.1f} [{100 * lo:+.1f}, {100 * hi:+.1f}]':>27}   {verdict}")
print("\nPaper (medical, RAG vs best graph method): " + "; ".join(f"{t} {a:.1f} vs {b:.1f}" for t, (a, b) in PAPER.items()))
print("\nLecture message (Slides 33-38): with every question of both domains the intervals are narrow enough to settle"
      "\nthe question that notebook 9 left open. Read the verdict column by question type: the graph's indexing price is"
      "\njustified only for the types where it is marked 'graph better'.")

RESULTS SUMMARY — Notebook 14: GraphRAG-Bench, all questions, both domains, gpt-6-luna, answer correctness x100
domain   question type               n   vector   graph    graph - vector [95% CI]   verdict
medical  Fact Retrieval           1098     64.9    63.3          -1.6 [-3.2, -0.0]   vector better
medical  Complex Reasoning         509     55.0    53.8          -1.1 [-3.2, +0.9]   no resolvable difference
medical  Contextual Summarize      289     57.6    56.0          -1.6 [-4.0, +0.8]   no resolvable difference
medical  Creative Generation       166     45.5    43.6          -1.9 [-4.1, +0.3]   no resolvable difference
medical  ALL                      2062     59.9    58.4          -1.5 [-2.5, -0.4]   vector better
novel    Fact Retrieval            971     58.6    54.7          -3.9 [-6.0, -1.9]   vector better
novel    Complex Reasoning         610     45.8    48.7          +2.8 [+1.3, +4.4]   graph better
novel    Contextual Summarize      362     51.4    52.7          +1.3 

## Added after review: HippoRAG2-style retrieval

The graph retriever above links LLM-extracted *question entities* to the graph, gives only a small weight to dense
scores and hands the answerer **5** chunks plus 25 relation lines. A reviewer noted that it retrieved 6–14 points less
gold evidence than vector top-8, so it may understate what a graph retriever can do. This section adds the retriever
of notebook 11, which follows HippoRAG 2 more closely, on the **same extracted graph** (no new extraction):

- **Graph:** the entity–chunk graph from the extraction above, plus **synonym edges** between entity names with
  embedding similarity ≥ 0.9 (HippoRAG 2).
- **Seeds:** the question embedding is matched against every extracted relation ("source — relation — target"); the
  entities of the 5 best-matching relations are weighted by similarity, and 0.05 × the dense score of the 20 best
  chunks is added to the reset vector, as in notebook 11 and HippoRAG 2. (HippoRAG 2's LLM filter of the matched
  facts is not used.)
- **Output:** personalised PageRank (damping 0.5); the **8** highest-ranked chunks — the same budget as vector top-8,
  with no relation lines.

Answers use the same prompt, and scoring uses the same statement/judge pipeline and the same gold statements.

**Run mode of this section:** the medical answers and statements went through the Batch API; because the Batch queue took hours per stage, the remaining stages (medical judge, and novel answers, statements and judge) were sent as synchronous standard-tier calls (the cell below switches a batch run to synchronous calls with 32 workers) at twice the Batch price. Requests and the cache are identical either way; the ledger's *via Batch* column shows which calls used which tier.

In [10]:
# Rebuild the entity-chunk graph from the cached extraction (same requests -> replayed from the cache) and add
# synonym edges (embedding similarity >= 0.9 between entity names) plus relation embeddings for seeding.
for dom, d in D.items():
    res = llm.run(f"{dom}_extract", {f"c{i}": LLM.body([{"role": "system", "content": EXTRACT_SYS[dom]},
                                                         {"role": "user", "content": c}], max_tokens=4000, schema=Extraction)
                                      for i, c in enumerate(d["chunks"])}, parse=Extraction)
    n_c, ent = d["n_c"], {norm(nm): d["n_c"] + j for j, nm in enumerate(d["ent_name"])}
    rows, cols, facts = [], [], []
    for ci in range(n_c):
        ex = res[f"c{ci}"]["parsed"] or {"entities": [], "relations": []}
        for e in ex["entities"]:
            j = ent.get(norm(e["name"]))
            if j is not None:
                rows += [ci, j]; cols += [j, ci]
        for r in ex["relations"]:
            a, b = ent.get(norm(r["source"])), ent.get(norm(r["target"]))
            if a is None or b is None or a == b:
                continue
            rows += [a, b, ci, a, ci, b]; cols += [b, a, a, ci, b, ci]
            facts.append((a, b, ci, f"{r['source']} — {r['description']} — {r['target']}"))
    syn = 0
    for s in range(0, len(d["ent_name"]), 512):
        S = d["E"][s:s + 512] @ d["E"].T
        ii, jj = np.where(S >= 0.9)
        for a, b in zip(ii + s, jj):
            if a < b:
                rows += [n_c + a, n_c + b]; cols += [n_c + b, n_c + a]; syn += 1
    n = n_c + len(d["ent_name"])
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n, n)); A.data[:] = 1
    deg = np.asarray(A.sum(1)).ravel(); deg[deg == 0] = 1
    d["T2"], d["facts"] = (sp.diags(1 / deg) @ A).T.tocsr(), facts
    d["F"] = llm.embed(f"{dom}_facts", [f[3] for f in facts], cap=300)
    print(f"{dom}: {len(facts):,} relation facts, {syn:,} synonym edges")

def hippo_retrieve(d, qi, k=8):
    q = d["Q"][qi]
    dense = d["C"] @ q
    pers = np.zeros(d["T2"].shape[0])
    fs = d["F"] @ q
    for j in np.argsort(-fs)[:5]:
        a, b, _, _ = d["facts"][j]
        pers[a] += fs[j]; pers[b] += fs[j]
    for j in np.argsort(-dense)[:20]:
        pers[j] += 0.05 * max(dense[j], 0)
    x = ppr(d["T2"], pers)[:d["n_c"]]
    return list(np.argsort(-x)[:k])

for dom, d in D.items():
    d["hippo"] = []
    for qi in range(len(d["questions"])):
        ids = hippo_retrieve(d, qi)
        d["hippo"].append(("\n\n".join(f"[chunk {i}] {d['chunks'][i]}" for i in ids), ids))
    print(f"{dom}: HippoRAG2-style retrieval done for {len(d['questions']):,} questions")

medical: 10,492 relation facts, 781 synonym edges


novel: 44,270 relation facts, 790 synonym edges


medical: HippoRAG2-style retrieval done for 2,062 questions


novel: HippoRAG2-style retrieval done for 2,010 questions


In [11]:
if MODE == "batch":                               # remaining stages as synchronous calls (see the note above)
    llm.mode, llm.workers = "test", 32
# Answers, statements, judge and similarity for the new retriever (new cache stages; vector/graph stages unchanged).
for dom, d in D.items():
    reqs = {f"hippo|{q['id']}": LLM.body([{"role": "system", "content": ANSWER_SYS[dom]},
                                          {"role": "user", "content": f"Context:\n{d['hippo'][qi][0]}\n\nQuestion: {q['question']}"}],
                                         max_tokens=1000)
            for qi, q in enumerate(d["questions"])}
    d["ans"].update({k: v["text"] for k, v in llm.run(f"{dom}_answers_hippo", reqs).items()})
    st = llm.run(f"{dom}_statements_hippo", {f"hippo|{q['id']}": LLM.body([{"role": "user", "content": STMT + d["ans"][f"hippo|{q['id']}"]}],
                                                                         max_tokens=2000, schema=Statements)
                                             for q in d["questions"]}, parse=Statements)
    d["st"].update({k: (v["parsed"] or {"statements": []})["statements"] for k, v in st.items()})
    reqs = {}
    for q in d["questions"]:
        gold, a = d["st"][f"gold|{q['id']}"], d["st"][f"hippo|{q['id']}"]
        reqs[f"cls|hippo|{q['id']}"] = LLM.body([{"role": "user", "content":
            "Compare answer statements with ground-truth statements. TP: answer statement supported by the "
            "ground truth. FP: answer statement not supported by (or contradicting) the ground truth. FN: "
            "ground-truth statement missing from the answer. Return the statements in each list.\n\n"
            f"Question: {q['question']}\nAnswer statements: {json.dumps(a)}\nGround-truth statements: {json.dumps(gold)}"}],
            max_tokens=3000, schema=Classified)
        if q["question_type"] in ("Contextual Summarize", "Creative Generation") and gold:
            reqs[f"cov|hippo|{q['id']}"] = LLM.body([{"role": "user", "content":
                "For each reference fact, is it covered by the response? Answer 1 or 0 per fact, in order.\n\n"
                f"Question: {q['question']}\nResponse: {d['ans'][f'hippo|{q['id']}'][:3000]}\nReference facts: {json.dumps(gold)}"}],
                max_tokens=500, schema=Coverage)
    for k, v in llm.run(f"{dom}_judge_hippo", reqs).items():
        try:
            d["judge"][k] = json.loads(v["text"])
        except Exception:
            d["judge"][k] = None
    keys = [f"hippo|{q['id']}" for q in d["questions"]]
    A = llm.embed(f"{dom}_answer_vecs_hippo", [d["ans"][k] for k in keys])
    G = llm.embed(f"{dom}_gold_vecs", [q["answer"] for q in d["questions"]])
    d["sim"].update({k: float(A[i] @ G[i]) for i, k in enumerate(keys)})
    print(f"{dom}: {len(keys):,} HippoRAG2-style answers scored")

[medical_judge_hippo] 2,517 new requests -> test (0 cached)


medical: 2,062 HippoRAG2-style answers scored
[novel_answers_hippo] 2,009 new requests -> test (0 cached)


[novel_statements_hippo] 2,009 new requests -> test (0 cached)


[novel_judge_hippo] 2,438 new requests -> test (0 cached)


novel: 2,010 HippoRAG2-style answers scored


In [12]:
SUMMARY_H = {}
print(f"{'domain':<9}{'question type':<23}{'n':>6}{'vector':>9}{'hippo':>8}{'hippo - vector [95% CI]':>27}"
      f"{'hippo - graph [95% CI]':>27}{'coverage v / h':>17}")
for dom, d in D.items():
    for t in TYPES + ["ALL"]:
        qs = [q for q in d["questions"] if t == "ALL" or q["question_type"] == t]
        v = [correctness(d, "vec", q) for q in qs]; g = [correctness(d, "graph", q) for q in qs]
        h = [correctness(d, "hippo", q) for q in qs]
        dm, lo, hi = paired_boot(v, h); dg, lg, hg = paired_boot(g, h)
        cov = ""
        if t in ("Contextual Summarize", "Creative Generation"):
            cov = f"{np.nanmean([coverage(d, 'vec', q) for q in qs]):.0%} / {np.nanmean([coverage(d, 'hippo', q) for q in qs]):.0%}"
        SUMMARY_H[(dom, t)] = (len(qs), np.mean(v), np.mean(h), dm, lo, hi)
        print(f"{dom:<9}{t:<23}{len(qs):>6}{100 * np.mean(v):>9.1f}{100 * np.mean(h):>8.1f}"
              f"{f'{100 * dm:+.1f} [{100 * lo:+.1f}, {100 * hi:+.1f}]':>27}{f'{100 * dg:+.1f} [{100 * lg:+.1f}, {100 * hg:+.1f}]':>27}{cov:>17}")

print("\nGold-evidence recall (vector top-8 / graph top-5 / HippoRAG2-style top-8):")
for dom, d in D.items():
    for t in TYPES:
        r = {m: [] for m in ("vec", "graph", "hippo")}
        for qi, q in enumerate(d["questions"]):
            evs = q.get("evidence") or []
            evs = evs if isinstance(evs, list) else [evs]
            if q["question_type"] != t or not evs:
                continue
            for m in r:
                r[m].append(np.mean([ev_hit(d, e, d[m][qi][1]) for e in evs]))
        print(f"  {dom:<8} {t:<22} vector {np.mean(r['vec']):5.1%}   graph {np.mean(r['graph']):5.1%}   hippo {np.mean(r['hippo']):5.1%}")

L = llm.ledger()
print("\nCost of the added stages: " + ", ".join(f"{s} ${L[s]['usd']:.2f}" for s in sorted(L) if s.endswith("_hippo")) +
      f" | total ${sum(L[s]['usd'] for s in L if s.endswith('_hippo')):.2f}")

domain   question type               n   vector   hippo    hippo - vector [95% CI]     hippo - graph [95% CI]   coverage v / h
medical  Fact Retrieval           1098     64.9    66.0          +1.1 [-0.3, +2.4]          +2.6 [+1.1, +4.2]                 
medical  Complex Reasoning         509     55.0    55.9          +1.0 [-0.7, +2.7]          +2.1 [+0.1, +4.0]                 
medical  Contextual Summarize      289     57.6    58.6          +1.0 [-1.0, +3.1]          +2.6 [+0.3, +4.9]        81% / 77%
medical  Creative Generation       166     45.5    46.7          +1.2 [-0.9, +3.4]          +3.1 [+0.7, +5.6]        47% / 43%
medical  ALL                      2062     59.9    60.9          +1.0 [+0.1, +1.9]          +2.5 [+1.5, +3.6]                 
novel    Fact Retrieval            971     58.6    62.4          +3.8 [+2.2, +5.5]          +7.7 [+5.7, +9.7]                 


novel    Complex Reasoning         610     45.8    47.5          +1.6 [+0.2, +3.1]          -1.2 [-2.7, +0.3]                 


novel    Contextual Summarize      362     51.4    53.4          +2.0 [-0.2, +4.0]          +0.7 [-1.3, +2.7]        89% / 90%
novel    Creative Generation        67     30.3    31.5          +1.1 [-2.7, +4.9]          -1.0 [-3.8, +1.8]        32% / 33%
novel    ALL                      2010     52.5    55.2          +2.7 [+1.8, +3.7]          +3.5 [+2.3, +4.6]                 

Gold-evidence recall (vector top-8 / graph top-5 / HippoRAG2-style top-8):


  medical  Fact Retrieval         vector 97.9%   graph 92.0%   hippo 97.1%


  medical  Complex Reasoning      vector 91.2%   graph 84.2%   hippo 90.2%


  medical  Contextual Summarize   vector 92.2%   graph 82.8%   hippo 90.9%


  medical  Creative Generation    vector 60.5%   graph 46.1%   hippo 54.9%


  novel    Fact Retrieval         vector 91.6%   graph 79.8%   hippo 94.0%


  novel    Complex Reasoning      vector 93.1%   graph 85.8%   hippo 95.0%


  novel    Contextual Summarize   vector 84.6%   graph 75.6%   hippo 85.5%
  novel    Creative Generation    vector 66.6%   graph 52.9%   hippo 66.8%

Cost of the added stages: medical_answers_hippo $0.46, medical_judge_hippo $0.23, medical_statements_hippo $0.07, novel_answers_hippo $0.95, novel_judge_hippo $0.17, novel_statements_hippo $0.11 | total $1.99
